# 饲喂料量vs换料vs标准

## README

In [10]:
# ------------- 配置数据 ------------------
WEANING_WEIGHT = 6.5  # 断奶仔猪重

## 4栋2单元数据

In [24]:
# 读取数据
import pandas as pd
from feed_analysis.config.path import PATH_DATA, PATH_FEED_PROCESSED, PATH_FIGURE_HTML
from feed_analysis.config.coding_schema import STD_HEADER_NAME
from feed_analysis.feed_pipeline.utils.age import recalibrate_age

# -------------- 读取处理后的饲喂数据 ---------------
df_42_build = pd.read_excel(PATH_DATA / 'growth' / '育肥4-2单元喂食量-用于生长曲线拟合.xlsx', index_col=False)

# -------------- 读取标准日龄FCR数据 ------------------
df_hx_std = pd.read_excel(PATH_DATA / 'ori' / '汇兴牧业-标准日龄饲料.xlsx', header=1)   # 汇兴2026饲喂标注

# ------------ 添加日龄和料肉比---------------
df_42_build['day_fcr'] = df_42_build['age'].map(df_hx_std.set_index('日龄')['日料肉比'])
# 填补缺失值
df_42_build.loc[df_42_build['day_fcr'].isna(), 'day_fcr'] = df_42_build['day_fcr'].min()

# ----------- 添加日增重和估计体重等---------------
df_42_build['day_weight'] = df_42_build['avg_food_kg'] / df_42_build['day_fcr']     # 日增重（kg）
df_42_build['weight_cumsum'] = df_42_build['day_weight'].cumsum()                   # 累计日增重（kg）
df_42_build['weight'] = df_42_build['weight_cumsum'] + WEANING_WEIGHT               # 总计重量
df_42_build['day_pct_weight'] = df_42_build['day_weight'] / df_42_build['weight']     # 日增重百分比


In [25]:
from itables import show
show(df_42_build)

Loading ITables v2.6.2 from the internet... (need help?)


总趋势变化

In [5]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'avg_food_kg', '育肥4-2单元料头均值', color='red')

In [6]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'day_weight', '育肥4-2单元日增重', color='red')

In [7]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'day_pct_weight', '育肥4-2单元日增重百分比', color='red')

In [8]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'day_weight', '育肥4-2单元日增重', color='red')

### 日龄-日增重拟合 多项式模型

In [9]:
df_42_build.columns

Index(['age', 'Date', 'food_total_kg', 'stock_num', 'avg_food_kg', 'day_fcr',
       'day_weight', 'weight_cumsum', 'weight', 'day_pct_weight'],
      dtype='object')

In [10]:
from feed_analysis.growing_fit.log_log_fit import fit_poly_log_log, poly_log_pred

poly_log_model, poly, scaler = fit_poly_log_log(df=df_42_build, x_col="age", y_col="day_weight", degree=3)     # Log模型拟合(直接最小二乘)
# print(poly_log_params_42)

# 添加拟合结果
df_42_build['day_weight_fit'] = poly_log_pred(df_42_build['age'].values, poly_log_model, poly, scaler)  # 拟合结果
df_42_build['weight_fit'] = df_42_build['day_weight_fit'].cumsum() + WEANING_WEIGHT # 体重拟合结果
df_42_build.dropna(inplace=True) 

# 保存拟合结果
df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-poly3-log拟合.xlsx', index=False)

# 扩展模型
import numpy as np
df_42_extended = pd.DataFrame({'age': np.arange(21, 180, 1)})
df_42_extended['day_weight_fit'] = poly_log_pred(df_42_extended['age'].values, poly_log_model, poly, scaler)  # 拟合结果
df_42_extended['weight_fit'] = df_42_extended['day_weight_fit'].cumsum() + WEANING_WEIGHT # 体重拟合结果
df_42_extended.dropna(inplace=True) 
# df_42_extended.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-poly2-log拟合(延长).xlsx', index=False)


                            OLS Regression Results                            
Dep. Variable:             day_weight   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.917
Method:                 Least Squares   F-statistic:                     407.6
Date:                Wed, 21 Jan 2026   Prob (F-statistic):           1.01e-58
Time:                        09:08:07   Log-Likelihood:                 194.33
No. Observations:                 112   AIC:                            -380.7
Df Residuals:                     108   BIC:                            -369.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6042      0.004    147.112      0.0

In [11]:
# ------------- 模型拟合评估 -------------
# 体重
from feed_analysis.growing_fit.metrics import get_r2, get_rmse, get_mape
r2_weight = get_r2(df_42_build['weight'], df_42_build['weight_fit'])
rmse_weight = get_rmse(df_42_build['weight'], df_42_build['weight_fit'])
mape_weight = get_mape(df_42_build['weight'], df_42_build['weight_fit'])
# 日增重
r2_day_weight = get_r2(df_42_build['day_weight'], df_42_build['day_weight_fit'])
rmse_day_weight = get_rmse(df_42_build['day_weight'], df_42_build['day_weight_fit'])
mape_day_weight = get_mape(df_42_build['day_weight'], df_42_build['day_weight_fit'])

print(f"体重拟合评估：R2={r2_weight:.4f}, RMSE={rmse_weight:.4f}, MAPE={mape_weight:.4f}")
print(f"日增重拟合评估：R2={r2_day_weight:.4f}, RMSE={rmse_day_weight:.4f}, MAPE={mape_day_weight:.4f}")

体重拟合评估：R2=0.9999, RMSE=0.3285, MAPE=1.0141
日增重拟合评估：R2=0.9019, RMSE=0.0760, MAPE=9.1245


In [12]:
# ------------- 总重量 拟合vs实际 -------------
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight'], mode='markers', name='实际重量', opacity=0.5))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight_fit'], mode='lines', name='拟合重量', opacity=1))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合') 
# fig.write_html(PATH_FIGURE_HTML/'育肥4栋2单元总重拟合.html')
fig.show()

In [13]:
# ------------- 日增重 拟合vs实际 -------------
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight'], mode='markers', name='实际日增重'))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight_fit'], mode='lines', name='拟合日增重'))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合')

## 3栋数据

三栋总体数据&存栏量

In [27]:
# 数据载入
import pandas as pd
from feed_analysis.config.path import PATH_DATA, PATH_FEED_PROCESSED
from feed_analysis.config.coding_schema import STD_HEADER_NAME
from feed_analysis.feed_pipeline.utils.age import recalibrate_age

# 读取喂食量数据 
df_31_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-1_build.parquet', engine='pyarrow')
df_32_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-2_build.parquet', engine='pyarrow')
df_33_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-3_build.parquet', engine='pyarrow')
df_34_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-4_build.parquet', engine='pyarrow')
# 读取存栏量数据
df_3_num = pd.read_excel(PATH_DATA / 'ori' / '育肥3栋总存栏数据.xlsx').rename(columns=STD_HEADER_NAME).sort_values(by=['Date'], ascending=True)
df_3_num['Date'] = df_3_num['Date']

# 计算头均值
df_3_build = pd.concat([df_31_build['food_total_kg'], df_32_build['food_total_kg'], df_33_build['food_total_kg'], df_34_build['food_total_kg']], axis=1)
df_3_build['total'] = df_3_build.sum(axis=1)
df_3_build.columns = ['3_1_feed', '3_2_feed', '3_3_feed', '3_4_feed', 'food_total_sum']

df_3_build['Date'] = df_31_build['Date']
df_3_build['age'] = df_31_build['age']
df_3_build['stock_num'] = df_3_build['Date'].map(df_3_num.set_index('Date')['stock_num'])
df_3_build['avg_food_kg'] = df_3_build['food_total_sum'] / df_3_build['stock_num']
# 取需要的字段
df_3_build = df_3_build[['Date', 'age', 'stock_num', 'food_total_sum', 'avg_food_kg']]

# 筛选日期
df_3_build = df_3_build.loc[df_3_build['Date'].between(pd.to_datetime('2025-08-28').date(), pd.to_datetime('2025-12-30').date()), :] 

# 添补需要的数据


# -------------- 读取标准日龄FCR数据 ------------------
df_hx_std = pd.read_excel(PATH_DATA / 'ori' / '汇兴牧业-标准日龄饲料.xlsx', header=1)   # 汇兴2026饲喂标注

# ------------ 添加日龄和料肉比---------------
df_3_build['day_fcr'] = df_3_build['age'].map(df_hx_std.set_index('日龄')['日料肉比'])
# 填补缺失值
df_3_build.loc[df_3_build['day_fcr'].isna(), 'day_fcr'] = df_3_build['day_fcr'].min()

# ----------- 添加日增重和估计体重等---------------
df_3_build['day_weight'] = df_3_build['avg_food_kg'] / df_3_build['day_fcr']        # 日增重（kg）
df_3_build['weight'] = df_3_build['day_weight'].cumsum() + WEANING_WEIGHT           # 总重（kg）

# 插补缺失值
from feed_analysis.feed_pipeline.utils.interpolate import interpolation
df_3_build = interpolation(df_3_build, index_col='age', )

In [28]:
df_3_build

,age,Date,stock_num,food_total_sum,avg_food_kg,day_fcr,day_weight,weight
0,32,2025-08-28 00:00:00.000000000,2430.0,525.6,0.216296,1.233333,0.175375,6.675375
1,32,2025-08-28 12:02:54.898785425,2430.0,525.6,0.216296,1.233333,0.175375,6.850751
2,33,2025-08-29 00:05:49.797570850,2430.0,703.6,0.289547,1.250000,0.231638,7.082389
3,33,2025-08-29 12:08:44.696356275,2430.0,703.6,0.289547,1.250000,0.231638,7.314026
4,34,2025-08-30 00:11:39.595141700,2430.0,860.4,0.354074,1.303030,0.271731,7.585758
...,...,...,...,...,...,...,...,...
243,154,2025-12-27 23:48:20.404858300,1855.0,7491.6,4.038598,3.221053,1.253813,219.447716
244,155,2025-12-28 11:51:15.303643724,1854.0,6313.2,3.405178,3.242105,1.050298,220.498015
245,155,2025-12-28 23:54:10.202429150,1854.0,6313.2,3.405178,3.242105,1.050298,221.548313
246,156,2025-12-29 11:57:05.101214574,1570.0,6109.6,3.891465,3.297872,1.179993,222.728306


In [29]:
from feed_analysis.growing_fit.log_log_fit import fit_poly_log_log, poly_log_pred
model_fit_3build, poly, scaler = fit_poly_log_log(df_3_build, x_col="age", y_col="day_weight", degree=3, show_metrics=False)        # R2 0.972

df_3_build['day_weight_fit'] = poly_log_pred(df_3_build['age'].values, model_fit_3build, poly, scaler)
df_3_build['weight_fit'] = df_3_build['day_weight_fit'].cumsum() + WEANING_WEIGHT   # 体重拟合结果
df_3_build.dropna(inplace=True) 


In [30]:
# ------------- 模型拟合评估 -------------
# 体重
from feed_analysis.growing_fit.metrics import get_r2, get_rmse, get_mape
r2_weight = get_r2(df_3_build['weight'], df_3_build['weight_fit'])
rmse_weight = get_rmse(df_3_build['weight'], df_3_build['weight_fit'])
mape_weight = get_mape(df_3_build['weight'], df_3_build['weight_fit'])
# 日增重
r2_day_weight = get_r2(df_3_build['day_weight'], df_3_build['day_weight_fit'])
rmse_day_weight = get_rmse(df_3_build['day_weight'], df_3_build['day_weight_fit'])
mape_day_weight = get_mape(df_3_build['day_weight'], df_3_build['day_weight_fit'])

print(f"体重拟合评估：R2={r2_weight:.4f}, RMSE={rmse_weight:.4f}, MAPE={mape_weight:.4f}")
print(f"日增重拟合评估：R2={r2_day_weight:.4f}, RMSE={rmse_day_weight:.4f}, MAPE={mape_day_weight:.4f}")

体重拟合评估：R2=0.9993, RMSE=1.7493, MAPE=1.7408
日增重拟合评估：R2=0.9131, RMSE=0.0634, MAPE=6.0675


扩展模型

In [32]:
# 扩展模型
import numpy as np
df_3_extended = pd.DataFrame({'age': np.arange(21, 180, 1)})
df_3_extended['day_weight_fit'] = poly_log_pred(df_3_extended['age'].values, model_fit_3build, poly, scaler)  # 拟合结果
df_3_extended['weight_fit'] = df_3_extended['day_weight_fit'].cumsum() + WEANING_WEIGHT # 体重拟合结果
df_3_extended.dropna(inplace=True) 
df_3_extended.to_excel(PATH_DATA / 'growth' / '汇兴3栋生长数据-poly2-log拟合(延长).xlsx', index=False)


In [ ]:
# ------------- 总重量 拟合vs实际 -------------
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['weight'], mode='markers', name='实际重量', opacity=0.5))
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['weight_fit'], mode='lines', name='拟合重量', opacity=1))
df_3_build
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合') 
fig.show()

In [ ]:
# ------------- 日增重 拟合vs实际 -------------
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['day_weight'], mode='markers', name='实际日增重'))
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['day_weight_fit'], mode='lines', name='拟合日增重'))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥3栋日增重拟合')